### Recommend using NVIDIA RTX A4000

In [1]:
# 0) ENV REPAIR & PINS

# Remove unneeded packages
%pip -q uninstall -y tensorflow tensorboard tensorboard-data-server tensorboard-plugin-wit ml-dtypes || true

# Repin ONNX and friends to stable combo for this repo
%pip -q uninstall -y onnx || true
%pip -q install "protobuf==3.20.3" "tensorboard==2.12.3" "wandb==0.15.10" "onnx==1.17.0" onnxruntime onnxscript onnx-graphsurgeon onnxconverter-common roboflow pickleshare

# Quiet logs
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print("✅Environment pinned. Now restart Kernel and run the next cell.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✅Environment pinned. Now restart Kernel and run the next cell.


In [1]:
# 1) CLONE YOLOv7 (SnapML fork) and install reqs
import pathlib, subprocess, sys, os, urllib.request

repo_url = "https://github.com/hartwoolery/yolov7"
root = pathlib.Path("/notebooks") if pathlib.Path("/notebooks").exists() else pathlib.Path.cwd()
repo = root / "yolov7"

if not repo.exists():
    print("Cloning YOLOv7 (SnapML fork)…")
    subprocess.run(["git", "clone", repo_url, str(repo)], check=True)
else:
    print("YOLOv7 repo already exists at", repo)

%cd {repo}
!git fetch -q
!git checkout -q export-snapml

# Install repo requirements
!pip -q install -r requirements.txt

# Download tiny weights
pt = pathlib.Path("yolov7-tiny.pt")
if not pt.exists():
    urllib.request.urlretrieve(
        "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt",
        str(pt)
    )
print("✅YOLOv7 ready")

YOLOv7 repo already exists at /notebooks/yolov7
/notebooks/yolov7
error: pathspec 'export-snapml' did not match any file(s) known to git
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.1.1+cu121 requires torch==2.1.1, but you have torch 2.9.0 which is incompatible.
✅YOLOv7 ready


In [2]:
# 2) ROBOFLOW DOWNLOAD
import os
from getpass import getpass
from roboflow import Roboflow

# Ask if not set
api_key = os.environ.get("ROBOFLOW_API_KEY", "").strip()
if not api_key:
    api_key = getpass("Enter your Roboflow API key (input hidden): ").strip()
    os.environ["ROBOFLOW_API_KEY"] = api_key

WORKSPACE = "ai-minor"
PROJECT   = "rubiks-cube-detector"
VERSION   = 4

try:
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)
    dataset = version.download("yolov7")
    DATASET_DIR = dataset.location
    print("✅Dataset at:", DATASET_DIR)
except Exception as e:
    print("❌ Roboflow download failed.")
    print("- Check API key.")
    print(f"- Check workspace/project/version: {WORKSPACE}/{PROJECT} v{VERSION}")
    print("- Error:", repr(e))
    raise

Enter your Roboflow API key (input hidden):  ········


loading Roboflow workspace...
loading Roboflow project...
✅Dataset at: /notebooks/yolov7/Rubiks-cube-detector-4


In [3]:
# 3) PATCH data.yaml TO ABSOLUTE PATHS
from pathlib import Path
import yaml, os

p = Path(DATASET_DIR)
with open(p/"data.yaml") as f:
    y = yaml.safe_load(f)

y["train"] = str(p/"train/images")
y["val"]   = str(p/"valid/images")
if (p/"test/images").exists():
    y["test"] = str(p/"test/images")

# Check class count matches names
if isinstance(y.get("names"), (list, tuple)):
    y["nc"] = len(y["names"])

with open(p/"data.yaml", "w") as f:
    yaml.safe_dump(y, f, sort_keys=False)

print("✅YAML fixed")
print("train:", y["train"])
print("val  :", y["val"])
print("test :", y.get("test"))
print("nc   :", y.get("nc"), "names:", y.get("names"))

✅YAML fixed
train: /notebooks/yolov7/Rubiks-cube-detector-4/train/images
val  : /notebooks/yolov7/Rubiks-cube-detector-4/valid/images
test : /notebooks/yolov7/Rubiks-cube-detector-4/test/images
nc   : 2 names: ['Scrambled', 'Solved']


In [8]:
# 4) TRAIN (FP32)

# Disable W&B
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

!python -u train.py \
  --data "{DATASET_DIR}/data.yaml" \
  --cfg cfg/training/yolov7-tiny.yaml \
  --weights yolov7-tiny.pt \
  --img 224 224 \
  --batch-size 32 \
  --epochs 50 \
  --workers 2 \
  --name rubiks_det_224 \
  --device 0 \
  --hyp data/hyp.scratch.tiny.yaml

YOLOR 🚀 1f54114 torch 2.9.0+cu128 CUDA:0 (NVIDIA RTX A4000, 16101.75MB)

Namespace(weights='yolov7-tiny.pt', cfg='cfg/training/yolov7-tiny.yaml', data='/notebooks/yolov7/Rubiks-cube-detector-4/data.yaml', hyp='data/hyp.scratch.tiny.yaml', epochs=50, batch_size=32, img_size=[224, 224], rect=False, resume=False, nosave=False, notest=False, noautoanchor=False, evolve=False, bucket='', cache_images=False, image_weights=False, device='0', multi_scale=False, single_cls=False, adam=False, sync_bn=False, local_rank=-1, workers=2, project='runs/train', entity=None, name='rubiks_det_224', exist_ok=False, quad=False, linear_lr=False, label_smoothing=0.0, upload_dataset=False, bbox_interval=-1, save_period=-1, artifact_alias='latest', freeze=[0], v5_metric=False, world_size=1, global_rank=-1, save_dir='runs/train/rubiks_det_224', total_batch_size=32)
tensorboard: Start with 'tensorboard --logdir runs/train', view at http://localhost:6006/
hyperparameters: lr0=0.01, lrf=0.01, momentum=0.937, weight

In [9]:
# 5) LOCATE RUN
import glob, os
exp_dir = sorted(glob.glob("runs/train/rubiks_det_224*"))[-1]
print("Using:", exp_dir)
print("Weights:", os.listdir(os.path.join(exp_dir, "weights")))

# Run detection preview on validation images
!python detect.py \
  --img-size 224 \
  --weights {exp_dir}/weights/best.pt \
  --conf 0.35 \
  --source "{DATASET_DIR}/valid/images"

# Open runs/detect/exp*/ to view a few outputs

Using: runs/train/rubiks_det_224
Weights: ['best.pt', 'epoch_049.pt', 'epoch_048.pt', 'epoch_045.pt', 'epoch_000.pt', 'last.pt', 'epoch_024.pt', 'epoch_046.pt', 'epoch_047.pt', 'init.pt']
Namespace(weights=['runs/train/rubiks_det_224/weights/best.pt'], source='/notebooks/yolov7/Rubiks-cube-detector-4/valid/images', img_size=224, conf_thres=0.35, iou_thres=0.45, device='', view_img=False, save_txt=False, save_conf=False, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project='runs/detect', name='exp', exist_ok=False, no_trace=False)
YOLOR 🚀 1f54114 torch 2.9.0+cu128 CUDA:0 (NVIDIA RTX A4000, 16101.75MB)

Fusing layers... 
IDetect.fuse
/usr/local/lib/python3.11/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Model Summ

## Lens Studio doesn't allow ONNX over 10MB and the importer doesn’t support ONNX quantization ops

In [10]:
# 6) FP32 SNAPML EXPORT (NO SIMPLIFY, NO QUANT)
!python export.py \
  --weights {exp_dir}/weights/best.pt \
  --grid \
  --export-snapml \
  --img-size 224 224 \
  --max-wh 224

Namespace(weights='runs/train/rubiks_det_224/weights/best.pt', img_size=[224, 224], batch_size=1, dynamic=False, dynamic_batch=False, grid=True, end2end=False, max_wh=224, topk_all=100, iou_thres=0.45, conf_thres=0.25, device='cpu', simplify=False, include_nms=False, fp16=False, int8=False, export_snapml=True)
YOLOR 🚀 1f54114 torch 2.9.0+cu128 CPU

Fusing layers... 
IDetect.fuse
/usr/local/lib/python3.11/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Model Summary: 208 layers, 6010302 parameters, 0 gradients, 13.0 GFLOPS
Export SnapML

Starting TorchScript export with torch 2.9.0+cu128...
TorchScript export success, saved as runs/train/rubiks_det_224/weights/best.torchscript.pt
CoreML export failure: No module named 'coremltools'

Starting Torch

In [11]:
# 7) VERIFY ONNX
import glob, os, onnxruntime as ort
onnx_path = sorted(glob.glob(f"{exp_dir}/weights/*.onnx"))[-1]
print("ONNX:", onnx_path, "size:", os.path.getsize(onnx_path)//1024, "KB")

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
print("Outputs:", [(o.name, o.shape) for o in sess.get_outputs()])

print("✅SnapML FP32 ONNX ready")

ONNX: runs/train/rubiks_det_224/weights/best.onnx size: 23650 KB
Outputs: [('output', [1, 21, 28, 28]), ('sigmoid_1', [1, 21, 14, 14]), ('sigmoid_2', [1, 21, 7, 7])]
✅SnapML FP32 ONNX ready
